# CS336 Assignment 1 (basics): Building a Transformer LM


## Problem (unicode1): Understanding Unicode (1 point)


#### (a) What Unicode character does chr(0) return?


`chr(0)` returns the Unicode character U+0000, which is known as the null character (often abbreviated as NUL).

Originally, it used the end of string in C-style strings.

It's non-printable control character, and typically has no visible representation.


In [19]:
chr(0)
print(chr(0))

 


#### (b) How does this character’s string representation (**repr**()) differ from its printed representation?


`__repr__() ` method returns representation of the character or object which internally used in Python.

On the other hand, `__str__()` method returns visual reprentation of the character or object which human can read.


#### (c) What happens when this character occurs in text?


In [20]:
chr(0)
print(chr(0))
"this is a test" + chr(0) + "string"
print("this is a test" + chr(0) + "string")

 
this is a test string


In C style strings, `chr(0)` is used to terminate the string.

However in Python, `chr(0)` is treated as a regular character, NUL.


## Problem (unicode2): Unicode Encodings (3 points)


#### (a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32?


UTF-8 can treat ASCII in one byte. However, UTF-16 treats ASCII 2 bytes and UTF-32 treats ASCII 4 bytes. There is a space efficiency.


In [21]:
print(len("a".encode("utf-8")))
print(len("a".encode("utf-16le")))
print(len("a".encode("utf-16")))
print(len("a".encode("utf-32le")))
print(len("a".encode("utf-32")))

1
2
4
4
8


#### (b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect?


In [22]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

위 코드는 ASCII 집합에 들어가는 것만 제대로 표현할 수 있고, 아래와 같이 한글이나 한자 등 ASCII 집합에 속하지 않는 문자열은 표현할 수 없습니다.


In [23]:
decode_utf8_bytes_to_str_wrong("안녕하세요!".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xec in position 0: unexpected end of data

#### (c) Give a two byte sequence that does not decode to any Unicode character(s).


UTF-8은 가변 길이 인코딩이기 때문에 각 바이트의 비트 패턴에 따라 몇 바이트로 해석할지 규칙이 필요합니다. 이 규칙을 어긴 바이트 시퀀스는 디코딩 시 UnicodeDecodeError가 발생합니다. 예를 들어, overlong encoding이나 잘못된 continuation byte 등이 해당됩니다.

반면 UTF-32는 고정 길이 4바이트 인코딩이므로 바이트 단위로 나눌 필요 없이 4바이트씩 끊어서 바로 디코딩이 가능합니다. 그러나 모든 4바이트 조합이 유효한 유니코드 문자로 해석되는 것은 아닙니다. 예를 들어, surrogate 범위(U+D800–U+DFFF)나 유효 범위를 벗어난 값은 UTF-32에서도 UnicodeDecodeError를 일으킬 수 있습니다.


아래 표와 같이 첫번째 바이트 값에 따라 어디까지가 한 문자인지 알 수 있습니다.

| 첫 바이트 | 문자 크기    |
| --------- | ------------ |
| 0 ~ 127   | 1 byte       |
| 128 ~ 191 | x            |
| 192 ~ 193 | x (Overlong) |
| 194 ~ 223 | 2 bytes      |
| 224 ~ 239 | 3 bytes      |
| 240 ~ 247 | 4 bytes      |
| 248 ~ 255 | x            |


In [10]:
b"\xc1\x00".decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc1 in position 0: invalid start byte

## Problem (train_bpe): BPE Tokenizer Training (15 points)


In [ ]:
training_text = """
Whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house (hol' up)
I said certified freak, seven days a week
Wet ass pussy, make that pullout game weak, woo! (Ah)
Yeah, yeah, yeah, yeah
Yeah, you fucking with some wet ass pussy
Bring a bucket and a mop for this wet ass pussy
Give me everything you got for this wet ass pussy
Beat it up, nigga, catch a charge
Extra large, and extra hard
Put this pussy right in yo' face
Swipe your nose like a credit card
Hop on top, I want a ride
I do a kegel while it's inside
Spit in my mouth, look at my eyes
This pussy is wet, come take a dive
Tie me up like I'm surprised
Let's role-play, I wear a disguise
I want you to park that big Mack truck right in this little garage
Make it cream, make me scream
Out in public, make a scene
I don't cook, I don't clean
But let me tell you, I got this ring (ayy, ayy)
Gobble me, swallow me, drip down the side of me (yeah)
Quick, jump out 'fore you let it get inside of me (yeah)
I tell him where to put it, never tell him where I'm 'bout to be
I run down on him 'fore I have a nigga running me
Talk yo' shit, bite your lip
Ask for a car while you ride that dick (while you ride that dick)
You ain't never gotta fuck him for a thing
He already made his mind up 'fore he came
Now get your boots and your coat for this wet ass pussy
He bought a phone just for pictures of this wet ass pussy
Pay my tuition just to kiss me on this wet ass pussy
Now make it rain if you wanna see some wet ass pussy
Look, I need a hard hitter, I need a deep stroke
I need a Henny drink, I need a weed smoker
Not a garden snake, I need a king cobra
With a hook in it, hope it lean over
He got some money, then that's where I'm headed
Pussy A-1, just like his credit
He got a beard, well, I'm tryna wet it
I let him taste it, and now he diabetic
I don't wanna spit, I wanna gulp
I wanna gag, I wanna choke
I want you to touch that lil' dangly thing that swing in the back of my throat
My head game is fire, punani Dasani
It's going in dry, and it's coming out soggy
I ride on that thing like the cops is behind me (yuh, ah)
I spit on his mic' and now he tryna sign me, woo
Your honor, I'm a freak bitch, handcuffs, leashes
Switch my wig, make him feel like he cheating
Put him on his knees, give him some' to believe in
Never lost a fight, but I'm looking for a beating
In the food chain, I'm the one that eat ya
If he ate my ass, he's a bottom feeder
Big D stand for big demeanor
I could make ya bust before I ever meet ya
If it don't hang, then he can't bang
You can't hurt my feelings, but I like pain
If he fuck me and ask, "Whose is it?"
When I ride the dick, I'ma spell my name, ah
Yeah, yeah, yeah
Yeah, you fucking with some wet ass pussy
Bring a bucket and a mop for this wet ass pussy
Give me everything you got for this wet ass pussy
Now from the top, make it drop, that's some wet ass pussy
Now get a bucket and a mop, that's some wet ass pussy
I'm talking WAP, WAP, WAP, that's some wet ass pussy
Macaroni in a pot, that's some wet ass pussy, huh
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house
There's some whores in this house

I'm the mother Fucking top madam
삼촌들 용돈 뺏는 깡패
그걸로 popping bottle 샴페인
사뿐히 밟아 빨간 carpet
남자들 숨을 참네
멋진 척 참 내
니 남친 나를 탐내
cuz im the motherfucking top madam
센 티만 계속 냈던
넌 정말 nothing
까고 보니 별 거 없어
넌 정말 nothing
어깨 힘 빼고 이제는 배워
완전히 정복해 여자판 나폴레옹
이제 씹을 거리 없지
단물 빠진 껌을
억지로 질겅 씹어 봐라
가출해 니 턱주가리
가슴에 턱 붙여 빨리
고개 끄덕
everybody bounce
im sick of them puss
puss puss
bitch you a freaking puss puss
puss puss
bitch you a freaking puss puss
im the mofucking top man
니네 비지니스는 다음에
imma make her super star man
cuz im a motherfuking top man
내가 뜰 줄 몰랐지
금방 질 줄 알았는데 모두 놀랐지
완전체로 돌아왔네
아이언맨은 피융
from the Rock Bottom 빼고
전부 fake shit
all the motherfuckers pissed off
왜냐면 그년 나를 위한 비서
아니 그년 나를 위한 시소
아니아니 그년 나를 위한 칙쇼
beast mode
우린 전국 돌며 니네들 삥 뜯어
매일이 payday
그 덤으로 highway 를 타며
학준 형과 vacay
만들어 방방곡곡
one hunit thousand babies
cuz you are my freaking Puss Puss
puss puss
cuz you are my freaking puss puss
puss puss
cuz you are my freaking puss puss
yes im the real bad man aye aye
yes im the real bad madam aye aye
im the motherfucking top man
im the motherfucking top madam
im the motherfucking top man
bitch you a fucking puss puss
puss puss
Bitch you a freaking puss puss
puss puss
Bitch you a freaking puss puss
bitch you a fucking puss
"""

{256: (32, 116), 257: (32, 97), 258: (105, 110), 259: (117, 115), 260: (256, 104), 261: (32, 109), 262: (114, 101), 263: (32, 119), 264: (32, 112), 265: (32, 104), 266: (259, 115), 267: (32, 115), 268: (264, 266), 269: (258, 103), 270: (105, 115), 271: (105, 116), 272: (111, 117), 273: (32, 235), 274: (32, 121), 275: (32, 102), 276: (101, 116), 277: (111, 109), 278: (104, 101), 279: (97, 110), 280: (32, 236), 281: (32, 258), 282: (32, 98), 283: (260, 270), 284: (99, 107), 285: (277, 101), 286: (274, 272), 287: (39, 115), 288: (267, 285), 289: (32, 99), 290: (107, 101), 291: (32, 73), 292: (111, 116), 293: (101, 97), 294: (32, 100), 295: (260, 101), 296: (111, 262), 297: (265, 111), 298: (105, 109), 299: (32, 108), 300: (278, 262), 301: (268, 121), 302: (257, 115), 303: (117, 284), 304: (97, 116), 305: (32, 103), 306: (302, 115), 307: (263, 276), 308: (263, 104), 309: (111, 112), 310: (111, 110), 311: (297, 259), 312: (311, 101), 313: (296, 115), 314: (261, 101), 315: (118, 101), 316: (

In [35]:
from functools import lru_cache
import regex
from collections import Counter
from typing import Dict, Tuple, List


def replace_pair_with_single(source: Tuple[int], target: Tuple[int], replacement: int) -> Tuple[int]:
    result = []
    i = 0
    while i < len(source):
        if i < len(source) - 1 and (source[i], source[i + 1]) == target:
            result.append(replacement)
            i += 2
        else:
            result.append(source[i])
            i += 1
    return tuple(result)


def count_byte_pair(unicode_counter: Dict[Tuple[int], int]) -> Dict[Tuple[int], int]:
    out = {}

    for codes, count in unicode_counter.items():
        for i in range(len(codes) - 1):
            byte_pair = codes[i : i + 2]
            out[byte_pair] = out.get(byte_pair, 0) + count

    return out


def get_most_frequent_byte_pair(byte_pair_counter: Dict[Tuple[int, int], int]) -> (Tuple[int, int], int):
    return max(byte_pair_counter.items(), key=lambda item: (item[1], item[0]))


def remove_special_tokens(training_text: str, special_tokens: List[str]) -> str:
    special_tokens_pattern = r"(" + "|".join(special_tokens) + ")"
    return regex.sub(special_tokens_pattern, "", training_text)


class BPE:
    special_tokens = []
    vocab = {}

    def __init__(self, special_tokens: List[str] = []):
        self.special_tokens = special_tokens

    def train(
        self,
        training_text: str,
        max_vocab_size: int = 1_000,
        min_freq: int = 3,
        max_iter: int = 10_000,
    ) -> Dict[int, Tuple[int, int]]:
        # base_token = 256
        # next_token = base_token + len(special_tokens)

        PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        pretokens = regex.findall(PAT, remove_special_tokens(training_text, self.special_tokens))
        pretoken_counter = Counter(pretokens)
        unicode_counter = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counter.items()}

        for _ in range(max_iter):
            if len(self.vocab) >= max_vocab_size:
                break

            byte_pair_counter = count_byte_pair(unicode_counter)
            (most_freq_byte_pair, most_freq) = get_most_frequent_byte_pair(byte_pair_counter)

            if most_freq < min_freq:
                break

            new_token = 256 + len(self.vocab)
            self.vocab[new_token] = most_freq_byte_pair

            unicode_counter = {
                replace_pair_with_single(codes, most_freq_byte_pair, new_token): count
                for codes, count in unicode_counter.items()
            }

        return self.vocab

    def encode(self, text: str) -> List[int]:
        pair_to_token = {pair: token for token, pair in self.vocab.items()}
        stack: List[int] = []

        for code in text.encode("utf-8"):
            stack.append(code)

            while len(stack) >= 2 and (stack[-2], stack[-1]) in pair_to_token:
                right = stack.pop()
                left = stack.pop()
                stack.append(pair_to_token[(left, right)])

        return stack

    def decode(self, tokens: List[int]) -> str:
        codes = [
            bytes([code])
            for token in tokens
            for code in (special_vocab[token] if token in special_vocab else self.__expand(token))
        ]
        return b"".join(codes).decode("utf-8")

    @lru_cache(maxsize=None)
    def __expand(self, token: int) -> Tuple[int]:
        if token not in self.vocab:
            return (token,)
        left, right = self.vocab[token]
        return self.__expand(left) + self.__expand(right)


special_vocab = {
    1000: "<pad>".encode("utf-8"),
    1001: "<unk>".encode("utf-8"),
    1002: "<s>".encode("utf-8"),
    1003: "</s>".encode("utf-8"),
    1004: "<mask>".encode("utf-8"),
}


test_text = """
I'm the mother Fucking top madam 
계집은 삼일한 자박꼼
"""

bpe = BPE()
bpe.train(training_text)
tokens = bpe.encode(test_text)
print(f"👀 - tokens: {tokens}")
text = bpe.decode(tokens)
print(f"👀 - text: {text}")

👀 - tokens: [10, 73, 39, 109, 32, 257, 101, 32, 109, 111, 257, 101, 114, 32, 70, 117, 99, 107, 105, 110, 103, 32, 116, 111, 112, 32, 109, 97, 100, 97, 109, 32, 10, 234, 179, 132, 236, 167, 145, 236, 157, 128, 32, 236, 130, 188, 236, 157, 188, 237, 149, 156, 32, 236, 158, 144, 235, 176, 149, 234, 188, 188, 10]
👀 - text: 
I'm the mother Fucking top madam 
계집은 삼일한 자박꼼



In [10]:
import regex, itertools
from collections import Counter
from functools import lru_cache
from typing import Dict, Tuple, List

# --------------------------------------------------
# 1. 스페셜 토큰 설정 & ID 매핑
# --------------------------------------------------
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]
SPECIAL_BASE = 256  # 0-255 : raw bytes
SPECIAL_TO_ID = {tok: SPECIAL_BASE + i for i, tok in enumerate(SPECIAL_TOKENS)}
print(f"👀 - SPECIAL_TO_ID: {SPECIAL_TO_ID}")
ID_TO_SPECIAL = {v: k for k, v in SPECIAL_TO_ID.items()}
FIRST_MERGE_ID = SPECIAL_BASE + len(SPECIAL_TOKENS)  # BPE 새 토큰 시작 ID


# --------------------------------------------------
# 2. 보조 유틸
# --------------------------------------------------
def replace_pair_with_single(t: Tuple[int], target: Tuple[int], repl: int) -> Tuple[int]:
    """코드 튜플 t에서 target 쌍을 repl 토큰 하나로 치환"""
    res, i = [], 0
    while i < len(t):
        if i < len(t) - 1 and (t[i], t[i + 1]) == target:
            res.append(repl)
            i += 2
        else:
            res.append(t[i])
            i += 1
    return tuple(res)


def get_byte_pair_counter(counter: Dict[Tuple[int], int]) -> Dict[Tuple[int, int], int]:
    """전체 코퍼스(바이트 코드 튜플 ↔ 빈도)에서 인접 쌍 빈도 계산"""
    out = {}
    for codes, freq in counter.items():
        for a, b in zip(codes, codes[1:]):
            out[(a, b)] = out.get((a, b), 0) + freq
    return out


def most_frequent(byte_pair_counter: Dict[Tuple[int, int], int]):
    """(빈도, 사전순) 기준 최대 쌍 리턴"""
    return max(byte_pair_counter.items(), key=lambda x: (x[1], x[0]))


# --------------------------------------------------
# 3. BPE 훈련
# --------------------------------------------------
def train_bpe(
    text: str,
    max_vocab: int = 1_000,
    min_freq: int = 3,
    max_iter: int = 10_000,
) -> Dict[int, Tuple[int, int]]:
    """
    스페셜 토큰을 보호하면서 BPE 병합 규칙을 학습.
    반환값: {새_토큰_ID: (left, right)}
    """
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    pretokens = regex.findall(PAT, text)

    # 스페셜 토큰은 병합 대상에서 제거
    pretoken_counter = Counter(t for t in pretokens if t not in SPECIAL_TOKENS)
    pretoken_unicode_counter = {tuple(tok.encode("utf-8")): freq for tok, freq in pretoken_counter.items()}

    token_to_pair: Dict[int, Tuple[int, int]] = {}
    next_id = FIRST_MERGE_ID
    for _ in range(max_iter):
        if len(token_to_pair) >= (max_vocab - FIRST_MERGE_ID):
            break
        pair_counter = get_byte_pair_counter(pretoken_unicode_counter)
        if not pair_counter:
            break
        (best_pair, best_freq) = most_frequent(pair_counter)
        if best_freq < min_freq:
            break

        token_to_pair[next_id] = best_pair
        pretoken_unicode_counter = {
            replace_pair_with_single(codes, best_pair, next_id): freq
            for codes, freq in pretoken_unicode_counter.items()
        }
        next_id += 1
    return token_to_pair  # ── BPE merge rules


# --------------------------------------------------
# 4. 바이트 시퀀스 → BPE 토큰 (merge-with-retie)
# --------------------------------------------------
def merge_encode(byte_ids: List[int], token_to_pair: Dict[int, Tuple[int, int]]) -> List[int]:
    pair_to_token = {pair: tok for tok, pair in token_to_pair.items()}
    stack: List[int] = []
    for code in byte_ids:
        stack.append(code)
        while len(stack) >= 2 and (stack[-2], stack[-1]) in pair_to_token:
            r = stack.pop()
            l = stack.pop()
            stack.append(pair_to_token[(l, r)])
    return stack


# --------------------------------------------------
# 5. 전체 인코더 (스페셜 토큰 처리 포함)
# --------------------------------------------------
SPECIAL_PATTERN = regex.compile("|".join(map(regex.escape, SPECIAL_TOKENS)))


def encode(text: str, token_to_pair: Dict[int, Tuple[int, int]]) -> List[int]:
    """
    문자열 → [토큰 ID]
    ① 스페셜 토큰 구간을 먼저 분리
    ② 일반 텍스트 구간은 UTF-8 바이트 → BPE 병합
    """
    ids: List[int] = []
    segments = regex.split(f"({SPECIAL_PATTERN.pattern})", text)
    for seg in segments:
        if not seg:
            continue
        if seg in SPECIAL_TO_ID:  # 스페셜 토큰
            ids.append(SPECIAL_TO_ID[seg])
        else:  # 일반 텍스트
            byte_ids = list(seg.encode("utf-8"))
            ids.extend(merge_encode(byte_ids, token_to_pair))
    return ids


# --------------------------------------------------
# 6. 디코더
# --------------------------------------------------
def decode(token_ids: List[int], token_to_pair: Dict[int, Tuple[int, int]]) -> str:
    # token_to_pair 는 외부 스코프에 그대로 두고,
    # expand() 는 tok 하나만 받도록 변경
    @lru_cache(maxsize=None)
    def expand(tok: int) -> Tuple[int, ...]:
        if tok not in token_to_pair:  # 더 이상 분해할 게 없으면
            return (tok,)
        left, right = token_to_pair[tok]
        return expand(left) + expand(right)

    parts, buf = [], []

    def flush():
        if buf:
            parts.append(bytes(buf).decode("utf-8", errors="replace"))
            buf.clear()

    for tid in token_ids:
        if tid in ID_TO_SPECIAL:  # 스페셜 토큰
            flush()
            parts.append(ID_TO_SPECIAL[tid])
        else:  # 일반 토큰 → 바이트 확장
            buf.extend(expand(tid))

    flush()
    return "".join(parts)


# --------------------------------------------------
# 7. 사용 예시
# --------------------------------------------------
if __name__ == "__main__":
    training_text = """
    There's some whores in this house,
    I'm the mother Fxxking top madam.
    """

    # ① BPE 병합 규칙 학습
    token_to_pair = train_bpe(training_text, max_vocab=2_000, min_freq=2)

    # ② 인코딩
    test_text = "<s>Hello <mask> 쎅쓰 보지 자지 world!</s>"
    encoded = encode(test_text, token_to_pair)
    print("Encoded →", encoded)

    # ③ 디코딩 (검증)
    decoded = decode(encoded, token_to_pair)
    print("Decoded →", decoded)

👀 - SPECIAL_TO_ID: {'<pad>': 256, '<unk>': 257, '<s>': 258, '</s>': 259, '<mask>': 260}
Encoded → [258, 72, 101, 108, 108, 111, 32, 260, 32, 236, 142, 133, 236, 147, 176, 32, 235, 179, 180, 236, 167, 128, 32, 236, 158, 144, 236, 167, 128, 32, 119, 111, 114, 108, 100, 33, 259]
Decoded → <s>Hello <mask> 쎅쓰 보지 자지 world!</s>
